# Урок 19. Первая модель и честная проверка: train и test

**Дата:** 28.01.2027 · **60 минут** · 8–9 класс  
**Модуль 4.** Машинное обучение (уроки 17–25)  
**Опирается на:** урок 17 (X, y, классификация), урок 4 (условия `if`), урок 10 (pandas).

Сегодня наконец обучим настоящую модель машинного обучения — **дерево решений**. И сразу столкнёмся с главной ловушкой: модель может показывать отличный результат на знакомых данных и ошибаться на новых. Научимся проверять модель **честно** — на данных, которых она не видела.

## Цели урока

К концу занятия ты сможешь:

1. Обучать модель в sklearn по схеме **создать → `fit` → `predict`**.
2. Объяснять, как дерево решений принимает решение.
3. Измерять качество долей верных ответов (**accuracy**).
4. Делить данные на **обучающую (train)** и **тестовую (test)** части.
5. Объяснять, почему качество на train обманчиво.

## План на 60 минут

| Минуты | Что делаем |
|-------:|------------|
| 0–5 | От ручного правила к модели |
| 5–20 | sklearn, дерево решений, первое предсказание |
| 20–35 | Ловушка «проверки на учебнике», train и test |
| 35–55 | Практика |
| 55–60 | Итог и домашнее задание |

## 1. Данные и задача

Задача из урока 17: по часам занятий, посещаемости и сну **заранее** предсказать, сдаст ли ученик итоговую работу. Это бинарная классификация. Возьмём побольше учеников — 200.

In [ ]:
import random
import pandas as pd

def make_students(n, seed):
    """Сгенерировать учебную таблицу учеников (как на уроке 12)."""
    random.seed(seed)
    rows = []
    for i in range(n):
        hours = random.randint(1, 14)
        attendance = min(100, max(20, round(50 + 3 * hours + random.gauss(0, 10))))
        score = min(100, max(0, round(20 + 4 * hours + 0.3 * attendance + random.gauss(0, 8))))
        rows.append({
            "класс": random.choice([8, 9]),
            "кружок": random.choice(["Python", "Роботы", "Игры"]),
            "часы": hours,
            "посещаемость": attendance,
            "сон": round(random.uniform(6, 9.5), 1),
            "балл": score,
        })
    return pd.DataFrame(rows)

df = make_students(200, seed=19)
df["сдал"] = df["балл"] >= 60

X = df[["часы", "посещаемость", "сон"]]
y = df["сдал"]
print("X:", X.shape, "| y:", y.shape)
print(y.value_counts())

## 2. Библиотека scikit-learn

**scikit-learn** (`sklearn`) — самая популярная библиотека «классического» машинного обучения. В ней десятки моделей, и все они работают по одной схеме:

```python
model = НазваниеМодели(настройки)   # 1. создать модель
model.fit(X, y)                     # 2. обучить на примерах с ответами
predictions = model.predict(X_new)  # 3. предсказать для новых объектов
```

`fit` означает «подогнать» — модель ищет закономерности в примерах. Выучишь эту схему один раз — сможешь пользоваться любой моделью.

Библиотека огромная, поэтому из неё берут только нужное — новой формой импорта:

```python
from sklearn.tree import DecisionTreeClassifier
```

Читается: «из раздела `tree` библиотеки `sklearn` возьми `DecisionTreeClassifier`». После этого его можно писать просто по имени, без `sklearn.tree.` впереди.

## 3. Дерево решений

**Дерево решений** — это цепочка вопросов «да / нет» про признаки, как `if / elif` из урока 4:

```text
часы >= 6?
├── да → посещаемость >= 70? → да: СДАСТ / нет: …
└── нет → …
```

На уроке 1 мы искали **один** порог по **одному** признаку на глаз (а на резервном уроке 16 — перебором в цикле). Дерево делает то же самое, но умнее: само выбирает, **какой признак** спросить, **какой порог** взять и **в каком порядке** задавать вопросы. Настройка `max_depth` — сколько вопросов подряд может задать дерево (глубина).

In [ ]:
from sklearn.tree import DecisionTreeClassifier

model = DecisionTreeClassifier(max_depth=1, random_state=0)
model.fit(X, y)
print("Модель обучена!")

Строка `model = DecisionTreeClassifier(...)` создаёт **ещё не обученную** модель с нашими настройками — как пустую тетрадь с подписанной обложкой. `model.fit(X, y)` её обучает: заполняет правилами по примерам.

Начнём с самого простого дерева — `max_depth=1`, то есть всего **один вопрос**. `random_state=0` — то же «зерно» случайности, что `random.seed`: при каждом запуске результат одинаковый.

### Что выучило дерево?

У дерева есть важное достоинство: его правила можно **прочитать**. Функция `export_text` печатает их в виде вопросов:

In [ ]:
from sklearn.tree import export_text

print(export_text(model, feature_names=["часы", "посещаемость", "сон"]))

Как читать: строка `|---` — это вопрос, под ней с отступом — ответ. `class: True` — предсказание «сдаст». Дерево само выбрало признак (часы, а не сон) и порог — то самое правило «если часов не меньше …», которое мы искали на уроке 1.

Модель — это просто **набор правил**, но правила нашёл не человек, а алгоритм, по примерам. Чем больше `max_depth`, тем больше вопросов и тем сложнее правила. Попробуй поставить `max_depth=3` и снова напечатать правила.

### Первое предсказание

Новый ученик: занимается 9 часов в неделю, посещаемость 80%, спит 7.5 часа. Признаки передаём **таблицей с теми же столбцами**, что при обучении.

In [ ]:
new_student = pd.DataFrame({"часы": [9], "посещаемость": [80], "сон": [7.5]})
print("Сдаст?", model.predict(new_student))

two_students = pd.DataFrame({"часы": [2, 12], "посещаемость": [45, 95], "сон": [8, 6.5]})
print("Двое новых:", model.predict(two_students))

`predict` возвращает **массив** — похожий на список набор ответов, по одному на каждую строку.

## 4. Качество модели: accuracy

Самый простой показатель качества — **accuracy** (доля верных ответов): сколько предсказаний совпало с правдой.

$$\text{accuracy} = \frac{\text{число верных ответов}}{\text{число всех ответов}}$$

Функция `accuracy_score(правильные_ответы, предсказания)` считает её.

In [ ]:
from sklearn.metrics import accuracy_score

pred = model.predict(X)
print("Accuracy на всех 200 учениках:", accuracy_score(y, pred))

## 5. Ловушка: проверка на учебнике

Мы проверили модель на тех же учениках, на которых она училась. Это как выдать на контрольной **те же задачи**, что разбирали в классе, да ещё с ответами на парте. Высокая оценка ничего не говорит о том, справится ли ученик с новыми задачами.

Посмотри, что будет, если разрешить дереву задавать очень много вопросов:

In [ ]:
deep_model = DecisionTreeClassifier(max_depth=20, random_state=0)
deep_model.fit(X, y)
print("Accuracy глубокого дерева на тех же учениках:", accuracy_score(y, deep_model.predict(X)))

Почти 100%! Неужели идеальная модель? Нет. Глубокое дерево задало столько вопросов, что фактически **запомнило каждого ученика**, включая случайный шум. Это зубрёжка, а не понимание. На новых учениках такое дерево ошибается — сейчас проверим.

## 6. Train и test: честная проверка

Решение: заранее **отложить** часть данных и **не показывать** её модели при обучении.

| Школа | Машинное обучение |
|-------|-------------------|
| задачи, которые разбирали в классе | **train** — обучающая выборка |
| контрольная с новыми задачами | **test** — тестовая выборка |
| подсмотреть ответы контрольной заранее | **утечка**: тест испорчен |

Функция `train_test_split` перемешивает строки и делит `X` и `y` на четыре части:

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

print("train:", X_train.shape, y_train.shape)
print("test: ", X_test.shape, y_test.shape)

Длинный вызов функции можно переносить на несколько строк, если перенос стоит **внутри скобок** — Python поймёт, что вызов продолжается. Слева от `=` четыре переменные: функция возвращает сразу четыре значения (как `plt.subplots` на уроке 13 — два).

Разберём параметры:

| Параметр | Смысл |
|----------|-------|
| `test_size=0.25` | 25% строк уходит в test, 75% — в train |
| `random_state=42` | одно и то же перемешивание при каждом запуске — у всех в классе одинаковый результат (42 — просто любое число, такая шутливая традиция программистов) |
| `stratify=y` | доля сдавших в train и test будет такой же, как во всех данных |

Порядок четырёх результатов фиксированный: **`X_train, X_test, y_train, y_test`**. Перепутать их — частая ошибка.

In [ ]:
print("Доля сдавших во всех данных:", round(y.mean(), 2))
print("в train:", round(y_train.mean(), 2), "| в test:", round(y_test.mean(), 2))

## 7. Обучаем на train — проверяем на test

In [ ]:
for depth in [3, 20]:
    m = DecisionTreeClassifier(max_depth=depth, random_state=0)
    m.fit(X_train, y_train)                                   # учим ТОЛЬКО на train
    acc_train = accuracy_score(y_train, m.predict(X_train))
    acc_test = accuracy_score(y_test, m.predict(X_test))      # проверяем на НОВЫХ
    print(f"глубина {depth:2}: train = {acc_train:.2f}, test = {acc_test:.2f}")

Напоминание (урок 12): `{acc_train:.2f}` в f-строке печатает число с двумя знаками после запятой, а `{depth:2}` отводит под число две позиции, чтобы строки шли ровно.

Глубокое дерево на train — «отличник», а на test — заметно хуже. Простое дерево на train скромнее, зато на новых данных работает лучше. **Честная оценка модели — та, что на test.** Явление «на train отлично, на новых плохо» называется **переобучением**; подробно разберём его на уроке 23.

## 8. Правила честной проверки

1. Test — это **сейф**. Модель не видит его при обучении.
2. Не подбирай настройки модели, многократно подглядывая в test: так test постепенно превращается в ещё один учебник. Как подбирать настройки честно — на уроке 23.
3. Всегда указывай `random_state`, чтобы результат можно было повторить.
4. Сообщай качество **на test**, а не на train.

## 9. Практика

### Практика 1. Один признак (7 минут)

Обучи дерево глубины 3 только на признаке `сон`. Какая accuracy на test? Сравни с моделью на трёх признаках и объясни результат, вспомнив разведку на уроке 12.

In [ ]:
# Практика 1
X_sleep = df[["сон"]]
# TODO: train_test_split (те же настройки), fit, accuracy на test

<details>
<summary>Решение</summary>

```python
Xs_train, Xs_test, ys_train, ys_test = train_test_split(
    X_sleep, y, test_size=0.25, random_state=42, stratify=y
)
m_sleep = DecisionTreeClassifier(max_depth=3, random_state=0)
m_sleep.fit(Xs_train, ys_train)
print("Только сон, test:", round(accuracy_score(ys_test, m_sleep.predict(Xs_test)), 2))
```
Accuracy намного ниже: сон почти не связан с результатом (корреляция близка к нулю). Модель не может найти закономерность, которой нет в данных.

Заметь: 0.72 — это ровно доля сдавших в test. Модель на одном сне, по сути, всем говорит «сдаст». Такую «ленивую» модель подробно разберём на уроке 21.
</details>

### Практика 2. Разные разбиения (7 минут)

Поменяй `random_state` в `train_test_split` на 1, 2, 3 и для каждого обучи дерево глубины 3. Как меняется accuracy на test? Почему она не одинаковая?

In [ ]:
# Практика 2

<details>
<summary>Решение</summary>

```python
for seed in [1, 2, 3]:
    Xa, Xb, ya, yb = train_test_split(X, y, test_size=0.25, random_state=seed, stratify=y)
    m = DecisionTreeClassifier(max_depth=3, random_state=0)
    m.fit(Xa, ya)
    print(seed, round(accuracy_score(yb, m.predict(Xb)), 2))
```
В test каждый раз попадают разные ученики — «лёгкие» или «трудные». В тесте всего 50 человек, поэтому одна-две ошибки заметно меняют долю. Чем больше тест, тем стабильнее оценка.
</details>

### Практика 3. Предсказание для себя (5 минут)

Обучи модель глубины 3 на `X_train, y_train` и предскажи результат для «себя»: сколько часов ты занимаешься, какая у тебя посещаемость и сон? Помни: модель обучена на выдуманных данных, к реальным людям её применять нельзя.

In [ ]:
# Практика 3

### Практика 4. Объясни своими словами (5 минут)

Почему 100% accuracy на train у глубокого дерева — **не** повод радоваться? Ответь 3–4 предложениями, используя слова «train», «test», «запомнило».

…

## Работа с ИИ-помощником

- «Объясни, как дерево решений выбирает, какой вопрос задать первым, — для 8 класса.»
- «Почему в `train_test_split` четыре результата и в каком они порядке?»
- «У меня accuracy на train 1.0, а на test 0.8. Что это значит?»

## Мини-глоссарий урока 19

| Термин | Смысл |
|--------|-------|
| **scikit-learn (sklearn)** | библиотека моделей машинного обучения |
| **`from … import …`** | взять из библиотеки одну нужную часть |
| **`fit` / `predict`** | обучить / предсказать |
| **Дерево решений** | модель — цепочка вопросов «да / нет» про признаки |
| **`max_depth`** | максимальная глубина дерева (сколько вопросов подряд) |
| **Accuracy** | доля верных предсказаний |
| **Train / test** | обучающая / тестовая (отложенная) выборка |
| **`stratify`** | одинаковая доля классов в train и test |
| **Переобучение** | модель запомнила обучающие данные и плохо работает на новых |

## Итог урока

1. Любая модель sklearn: **создать → `fit` → `predict`**.
2. Дерево решений — это правила `if`, которые алгоритм нашёл сам по примерам.
3. Качество проверяют **только на test**: на train модель может просто «зазубрить» ответы.

## Домашнее задание (20 минут)

1. Обучи деревья глубины 1, 2, 3, 5, 10 на `X_train` и запиши accuracy на train и test в таблицу.
2. Какая глубина лучше всего на test? Напиши 2–3 предложения о том, что происходит с train и test при росте глубины.

In [ ]:
# Домашнее задание

## Самопроверка

1. Назови три шага работы с любой моделью sklearn.
2. Как дерево решений связано с `if / elif`?
3. Что такое accuracy?
4. Зачем откладывать часть данных в test?
5. Что значит `stratify=y`?

**Следующий урок (04.02.2027):** классификация на практике — второй алгоритм (k ближайших соседей) и сравнение моделей.